# Model Serving — Endpoint Multi-Modelo com Roteamento (Model-Inside-Model)

Este notebook cria um endpoint de Model Serving que **hospeda dois modelos internamente** e **roteia a requisição** para o modelo mais adequado com base em um campo do body.

> **Padrão Model-Inside-Model**: ambos os modelos são empacotados como artefatos dentro de um único Custom PyFunc. O roteamento acontece dentro do `predict()`, sem depender do traffic split nativo do Databricks.

**Pré-requisito:** execute o notebook `3-Treinamento` antes para garantir que a tabela de treino e o modelo base existam no Unity Catalog.

**Lógica de roteamento:**
- `modelo_alvo = "conservador"` → ElasticNet (linear, regularizado)
- `modelo_alvo = "agressivo"` → Gradient Boosting (não-linear, complexo)
- `modelo_alvo = "auto"` ou ausente → roteamento automático baseado nas features

In [0]:
%pip install databricks-sdk==0.36.0 mlflow==2.19.0 scikit-learn==1.5.2
dbutils.library.restartPython()

In [0]:
# ============================================================
# Preencha apenas estas variáveis (mesmo padrão do 3-Treinamento)
# ============================================================
CATALOGO = ""
SCHEMA = ""
PREFIXO = ""

# Nomes derivados
identificador_schema = f"{CATALOGO}.{SCHEMA}"
router_model_name = f"{identificador_schema}.{PREFIXO}_credit_risk_router"
router_endpoint_name = f"{PREFIXO}-credit-risk-router"

## Treinar dois modelos especializados

Treinamos dois modelos com filosofias diferentes para o mesmo problema de risco de crédito:
- **Conservador (ElasticNet)**: modelo linear com regularização L1/L2, mais estável e interpretável
- **Agressivo (Gradient Boosting)**: modelo não-linear que captura padrões complexos, maior acurácia mas menos interpretável

In [0]:
import mlflow
import pandas as pd
import numpy as np
from mlflow.models import infer_signature
from mlflow import MlflowClient
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
import tempfile, os, pickle

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/{PREFIXO}_multi_model_routing")

# ============================================================
# Treinar dois modelos especializados para o roteamento
# ============================================================
df_treino = spark.table(f"{identificador_schema}.{PREFIXO}credit_risk_train_df").toPandas()
X = df_treino.drop(columns=["defaulted"])
y = df_treino["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo 1: Conservador (ElasticNet — linear, regularizado)
modelo_conservador = ElasticNet(alpha=0.5, l1_ratio=0.5, random_state=42, max_iter=1000)
modelo_conservador.fit(X_train, y_train)
print(f"Modelo Conservador (ElasticNet) — R²: {modelo_conservador.score(X_test, y_test):.4f}")

# Modelo 2: Agressivo (Gradient Boosting — não-linear, complexo)
modelo_agressivo = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
modelo_agressivo.fit(X_train, y_train)
print(f"Modelo Agressivo (GradientBoosting) — R²: {modelo_agressivo.score(X_test, y_test):.4f}")

# Salvar artefatos dos modelos
tmp_dir = tempfile.mkdtemp()

conservador_path = os.path.join(tmp_dir, "modelo_conservador.pkl")
with open(conservador_path, "wb") as f:
    pickle.dump(modelo_conservador, f)

agressivo_path = os.path.join(tmp_dir, "modelo_agressivo.pkl")
with open(agressivo_path, "wb") as f:
    pickle.dump(modelo_agressivo, f)

# Salvar colunas esperadas
colunas_path = os.path.join(tmp_dir, "columns.txt")
with open(colunas_path, "w") as f:
    f.write(",".join(X.columns.tolist()))

print(f"\nArtefatos salvos em: {tmp_dir}")
print(f"Features: {len(X.columns)} colunas")

## Definição do Custom PyFunc com roteamento

O `CreditRiskRouter` é um modelo customizado que:
1. Recebe a requisição com dados + campo `modelo_alvo`
2. Pré-processa (valida colunas, trata nulos, clipa outliers)
3. Roteia para o modelo correto com base no valor de `modelo_alvo`
4. Retorna `risk_score` + `modelo_utilizado` (transparência para o consumidor)

In [0]:
# ============================================================
# Custom PyFunc com roteamento inteligente
# ============================================================
class CreditRiskRouter(mlflow.pyfunc.PythonModel):
    """
    Endpoint multi-modelo com pré-processamento e roteamento.
    
    Recebe um campo 'modelo_alvo' no body que determina qual modelo usar:
    - 'conservador' → ElasticNet
    - 'agressivo'   → Gradient Boosting
    - ausente/outro → roteamento automático baseado nas features
    """

    def load_context(self, context):
        """Carrega ambos os modelos e metadados ao inicializar."""
        import pickle

        with open(context.artifacts["modelo_conservador"], "rb") as f:
            self.modelo_conservador = pickle.load(f)

        with open(context.artifacts["modelo_agressivo"], "rb") as f:
            self.modelo_agressivo = pickle.load(f)

        with open(context.artifacts["columns_file"], "r") as f:
            self.expected_columns = f.read().strip().split(",")

    def _preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        """Pré-processamento: validação, nulos, tipos."""
        feature_cols = [c for c in self.expected_columns]
        
        for col in feature_cols:
            if col not in df.columns:
                df[col] = 0.0

        df = df[feature_cols].copy()
        df = df.fillna(0)

        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
            df[col] = df[col].clip(-1e6, 1e6)

        return df

    def _decide_route(self, row_features: pd.Series) -> str:
        """
        Roteamento automático quando modelo_alvo não é informado.
        Lógica: se a média das features normalizadas > 0.3, usa agressivo.
        """
        mean_val = row_features.mean()
        return "agressivo" if mean_val > 0.3 else "conservador"

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """Roteia e prediz."""
        # Extrair coluna de roteamento (se existir)
        if "modelo_alvo" in model_input.columns:
            routes = model_input["modelo_alvo"].fillna("auto").values
            model_input = model_input.drop(columns=["modelo_alvo"])
        else:
            routes = ["auto"] * len(model_input)

        # Pré-processar features
        processed = self._preprocess(model_input)

        # Predizer linha a linha com roteamento
        predictions = []
        models_used = []

        for i in range(len(processed)):
            row = processed.iloc[[i]]
            route = routes[i]

            # Roteamento automático se necessário
            if route not in ("conservador", "agressivo"):
                route = self._decide_route(processed.iloc[i])

            # Invocar modelo correto
            if route == "conservador":
                pred = self.modelo_conservador.predict(row)[0]
            else:
                pred = self.modelo_agressivo.predict(row)[0]

            predictions.append(pred)
            models_used.append(route)

        return pd.DataFrame({
            "risk_score": predictions,
            "modelo_utilizado": models_used
        })


print("Classe CreditRiskRouter definida com sucesso!")

In [0]:
# ============================================================
# Registrar modelo com roteamento no Unity Catalog
# ============================================================

# Gerar signature — input inclui 'modelo_alvo' + features
df_input_example = X_test.head(3).copy()
df_input_example["modelo_alvo"] = ["conservador", "agressivo", "auto"]

# Output esperado
df_output_example = pd.DataFrame({
    "risk_score": [0.1, 0.8, 0.5],
    "modelo_utilizado": ["conservador", "agressivo", "agressivo"]
})

signature = infer_signature(df_input_example, df_output_example)

with mlflow.start_run(run_name="credit_risk_router"):
    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=CreditRiskRouter(),
        artifacts={
            "modelo_conservador": conservador_path,
            "modelo_agressivo": agressivo_path,
            "columns_file": colunas_path,
        },
        signature=signature,
        input_example=df_input_example,
        pip_requirements=[
            "scikit-learn==1.5.2",
            "mlflow==2.19.0",
            "pandas",
            "numpy",
        ],
        registered_model_name=router_model_name,
    )

# Definir alias "prod"
client_mlflow = MlflowClient()
versions = client_mlflow.search_model_versions(f"name='{router_model_name}'")
latest_version = str(max([int(v.version) for v in versions]))
client_mlflow.set_registered_model_alias(router_model_name, "prod", latest_version)

print(f"Modelo registrado: {router_model_name} v{latest_version} (alias: prod)")
print(f"URI: {model_info.model_uri}")

## Criação do endpoint de serving

Criamos um endpoint com **uma única entidade** que contém o PyFunc com ambos os modelos embutidos. O roteamento acontece internamente no `predict()`.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

try:
    print(f"Criando endpoint '{router_endpoint_name}' com roteamento multi-modelo...")
    w.serving_endpoints.create_and_wait(
        name=router_endpoint_name,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    name=f"{PREFIXO}-credit-router-entity",
                    entity_name=router_model_name,
                    entity_version=latest_version,
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                )
            ]
        ),
    )
    print(f"Endpoint '{router_endpoint_name}' criado e pronto!")
except Exception as e:
    if "already exists" in str(e).lower() or "RESOURCE_ALREADY_EXISTS" in str(e):
        print(f"Endpoint '{router_endpoint_name}' já existe. Atualizando...")
        w.serving_endpoints.update_config_and_wait(
            name=router_endpoint_name,
            served_entities=[
                ServedEntityInput(
                    name=f"{PREFIXO}-credit-router-entity",
                    entity_name=router_model_name,
                    entity_version=latest_version,
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                )
            ],
        )
        print(f"Endpoint '{router_endpoint_name}' atualizado!")
    else:
        raise e

## Teste de roteamento

Testamos enviando requisições com diferentes valores de `modelo_alvo`:
1. `"conservador"` → força ElasticNet
2. `"agressivo"` → força Gradient Boosting
3. `"auto"` → endpoint decide automaticamente com base nas features

In [0]:
import mlflow.deployments

# Preparar dados de teste com diferentes rotas
df_teste_router = X_test.head(6).copy()
df_teste_router["modelo_alvo"] = [
    "conservador",  # Força ElasticNet
    "conservador",  # Força ElasticNet
    "agressivo",    # Força Gradient Boosting
    "agressivo",    # Força Gradient Boosting
    "auto",         # Roteamento automático
    "auto",         # Roteamento automático
]

print("Requisição enviada ao endpoint:")
print(f"  - 2 registros com modelo_alvo='conservador'")
print(f"  - 2 registros com modelo_alvo='agressivo'")
print(f"  - 2 registros com modelo_alvo='auto' (roteamento automático)")

# Invocar o endpoint
deploy_client = mlflow.deployments.get_deploy_client("databricks")

response = deploy_client.predict(
    endpoint=router_endpoint_name,
    inputs={"dataframe_split": df_teste_router.to_dict(orient="split")},
)

print(f"\nResposta do endpoint:")
resultado_df = pd.DataFrame(response["predictions"] if "predictions" in response else response)
resultado_df.index = range(len(resultado_df))
print(resultado_df.to_string())

# Exibir resultado completo
resultado_final = pd.concat([
    df_teste_router[["modelo_alvo"]].reset_index(drop=True),
    resultado_df
], axis=1)

print(f"\n--- Resumo do roteamento ---")
print(resultado_final[["modelo_alvo", "modelo_utilizado", "risk_score"]].to_string(index=False))
display(spark.createDataFrame(resultado_final))

## Invocação direta de uma entidade específica (served_model_name)

Quando um endpoint tem **múltiplas entidades servidas** (traffic split), normalmente o Databricks roteia aleatoriamente por percentual.

Para **forçar** qual entidade chamar (ignorando o traffic split), use:
- **Databricks SDK**: parâmetro `served_model_name` em `query()`
- **REST API**: URL `/serving-endpoints/{endpoint}/served-models/{entidade}/invocations`

> **Nota:** No nosso caso (Model-Inside-Model), o roteamento é feito pelo campo `modelo_alvo` no body, pois temos apenas 1 entidade servida. O `served_model_name` seria útil se tivéssemos 2 entidades separadas no mesmo endpoint.

In [0]:
# ============================================================
# Exemplo: forçar invocação de uma entidade específica
# (útil quando há múltiplas entidades com traffic split)
# ============================================================

entidade_alvo = f"{PREFIXO}-credit-router-entity"

# Dados de exemplo
df_invoke = X_test.head(3).copy()
df_invoke["modelo_alvo"] = ["agressivo", "conservador", "auto"]

# MÉTODO 1: Databricks SDK — query() com served_model_name
print("=" * 60)
print("MÉTODO 1: Databricks SDK (served_model_name)")
print("=" * 60)

response_sdk = w.serving_endpoints.query(
    name=router_endpoint_name,
    served_model_name=entidade_alvo,  # <-- FORÇA a entidade específica
    dataframe_split={
        "columns": df_invoke.columns.tolist(),
        "data": df_invoke.values.tolist(),
    },
)
print(f"Endpoint: {router_endpoint_name}")
print(f"Entidade forçada: {entidade_alvo}")
print(f"Resposta: {response_sdk.predictions}")

# MÉTODO 2: REST API — URL direta para entidade
print(f"\n{'=' * 60}")
print("MÉTODO 2: REST API (URL direta para entidade)")
print("=" * 60)

import requests

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = spark.conf.get("spark.databricks.workspaceUrl")

url_direta = f"https://{host}/serving-endpoints/{router_endpoint_name}/served-models/{entidade_alvo}/invocations"
print(f"URL: {url_direta}")

response_rest = requests.post(
    url_direta,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"dataframe_split": {
        "columns": df_invoke.columns.tolist(),
        "data": df_invoke.values.tolist(),
    }},
)
print(f"Status: {response_rest.status_code}")
print(f"Resposta: {response_rest.json()}")